<a href="https://colab.research.google.com/github/dhruthirs/adversarial-resilience/blob/main/notebooks/cifar10_resnet18_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Evaluation: ResNet18 on CIFAR-10 under FGSM & PGD

This is your Objective 1 notebook: evaluate a real SOTA architecture on a real dataset, then attack it.

**Reuse plan:** this same notebook structure works for your other 2 architectures (MobileNetV2, ViT) — just swap the model definition cell (Step D) and re-run everything else unchanged. Keep results from each run in the shared results table at the bottom (Step I) so you build one master comparison across all models.

## Step A: Install the attack library

In [6]:
!pip install torchattacks -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.25.1 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.25.1 which is incompatible.
google-adk 2.3.0 requires requests<3,>=2.32.4, but you have requests 2.25.1 which is incompatible.
pysal 25.7 requires requests>=2.27, but you have requests 2.25.1 which is incompatible.
datasets 4.0.0 requires requests>=2.32.2, but you have requests 2.25.1 which is incompatible.
sphinx 8.2.3 requires requests>=2.30.0, but you have requests 2.25.1 which is incompatible.
yfinance 0.2.66 requires requests>=2.31, but you have requests 2.25.1 which is incompatible.
google-genai 2.10.0 requires requests<3.0.0,>=2.28.1, but you have requests 2.25.1 which is incompatible.
bigframes 2.42.0 requires requests>=2.27.1, but you have req

## Step B: Imports and device setup

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torchattacks
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Step C: Load CIFAR-10

60,000 color images (32x32), 10 classes: plane, car, bird, cat, deer, dog, frog, horse, ship, truck.
This is a meaningfully harder dataset than MNIST — real objects, real colors, more visual complexity.

In [8]:
!pip install datasets -q

from datasets import load_dataset
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

hf_dataset = load_dataset('uoft-cs/cifar10')

def to_tensor_dataset(split):
    images = np.stack([np.array(img) for img in split['img']])
    images = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    labels = torch.tensor(split['label'], dtype=torch.long)
    return TensorDataset(images, labels)

train_set = to_tensor_dataset(hf_dataset['train'])
test_set = to_tensor_dataset(hf_dataset['test'])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

classes = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f'Train samples: {len(train_set)}, Test samples: {len(test_set)}')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchattacks 3.5.1 requires requests~=2.25.1, but you have requests 2.34.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Train samples: 50000, Test samples: 10000


## Step D: Define ResNet18 (adapted for CIFAR-10's small 32x32 images)

Standard torchvision ResNet18 expects 224x224 ImageNet images. We modify the first layers so it works well on CIFAR-10's smaller images — this is the standard adaptation used across adversarial ML research code.

**To swap architectures later:** replace just this cell. For MobileNetV2: `torchvision.models.mobilenet_v2(num_classes=10)`. For a ViT, look up `timm` library's `vit_tiny_patch16_224` (needs image resizing to 224x224, more involved — ask me when you get there).

In [9]:
def build_resnet18_cifar():
    model = torchvision.models.resnet18(weights=None, num_classes=10)
    # swap the first conv layer: smaller kernel, no aggressive downsampling (since images are only 32x32)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()  # remove the maxpool that would shrink our already-small image too fast
    return model

model = build_resnet18_cifar().to(device)
print('Model ready. Total parameters:', sum(p.numel() for p in model.parameters()))

Model ready. Total parameters: 11173962


## Step E: Train it

This takes longer than MNIST (~1-2 minutes per epoch on Colab's free GPU). 10 epochs gets you roughly 75-80% clean accuracy — good enough for a mini project baseline. If you have time, more epochs = better accuracy, but diminishing returns after ~15-20.

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10  # increase later if you want higher clean accuracy; 10 is a reasonable baseline

model.train()
start = time.time()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} - loss: {running_loss/len(train_loader):.4f}')

print(f'Training done in {(time.time()-start)/60:.1f} minutes.')

Epoch 1/10 - loss: 1.1957
Epoch 2/10 - loss: 0.7234
Epoch 3/10 - loss: 0.5331
Epoch 4/10 - loss: 0.4036
Epoch 5/10 - loss: 0.3049
Epoch 6/10 - loss: 0.2171
Epoch 7/10 - loss: 0.1532
Epoch 8/10 - loss: 0.1131
Epoch 9/10 - loss: 0.0901
Epoch 10/10 - loss: 0.0727
Training done in 7.1 minutes.


## Step F: Measure CLEAN accuracy (no attack)

In [11]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

clean_acc = evaluate(model, test_loader)
print(f'Clean accuracy (no attack): {clean_acc:.2f}%')

Clean accuracy (no attack): 81.91%


## Step G: Run FGSM attack

CIFAR-10 images use the same 0-1 pixel scale as MNIST, so the same eps intuition applies. eps=8/255 (≈ 0.031) is the standard value used in most adversarial ML papers for CIFAR-10 — small enough to stay fairly imperceptible.

In [12]:
eps_fgsm = 8/255  # standard CIFAR-10 benchmark value (~0.031)

atk_fgsm = torchattacks.FGSM(model, eps=eps_fgsm)

model.eval()
correct, total = 0, 0
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    adv_images = atk_fgsm(images, labels)

    outputs = model(adv_images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

fgsm_accuracy = 100 * correct / total
print(f'Accuracy UNDER FGSM attack (eps={eps_fgsm:.3f}): {fgsm_accuracy:.2f}%')
print(f'FGSM Attack Success Rate (ASR): {100 - fgsm_accuracy:.2f}%')

Accuracy UNDER FGSM attack (eps=0.031): 2.06%
FGSM Attack Success Rate (ASR): 97.94%


## Step H: Run PGD attack

Standard CIFAR-10 PGD benchmark settings: eps=8/255, alpha=2/255, steps=10 (or 20 for a stronger/more thorough attack — used in most published papers).

In [13]:
eps_pgd = 8/255
alpha_pgd = 2/255
steps_pgd = 20

atk_pgd = torchattacks.PGD(model, eps=eps_pgd, alpha=alpha_pgd, steps=steps_pgd)

model.eval()
correct, total = 0, 0
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    adv_images = atk_pgd(images, labels)

    outputs = model(adv_images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

pgd_accuracy = 100 * correct / total
print(f'Accuracy UNDER PGD attack (eps={eps_pgd:.3f}): {pgd_accuracy:.2f}%')
print(f'PGD Attack Success Rate (ASR): {100 - pgd_accuracy:.2f}%')

Accuracy UNDER PGD attack (eps=0.031): 0.00%
PGD Attack Success Rate (ASR): 100.00%


## Step I: Results summary (record this in your shared team doc)

Copy this printed table into your team's results tracker every time you run this notebook for a new architecture.

In [14]:
print('=' * 55)
print('BASELINE RESULTS — Model: ResNet18 | Dataset: CIFAR-10')
print('=' * 55)
print(f'{"Condition":<20}{"Accuracy":<12}{"ASR":<10}')
print('-' * 55)
print(f'{"Clean":<20}{clean_acc:<12.2f}{"-":<10}')
print(f'{"FGSM (eps=8/255)":<20}{fgsm_accuracy:<12.2f}{100-fgsm_accuracy:<10.2f}')
print(f'{"PGD (eps=8/255)":<20}{pgd_accuracy:<12.2f}{100-pgd_accuracy:<10.2f}')
print('=' * 55)

BASELINE RESULTS — Model: ResNet18 | Dataset: CIFAR-10
Condition           Accuracy    ASR       
-------------------------------------------------------
Clean               81.91       -         
FGSM (eps=8/255)    2.06        97.94     
PGD (eps=8/255)     0.00        100.00    


## What to do next

1. Run this notebook fully once for ResNet18 (this file, as-is) — record the Step I results.
2. Duplicate this notebook, swap only Step D for MobileNetV2, re-run everything — record results again.
3. Repeat for your third architecture.
4. Once all 3 models have results, you have your full Objective 1 baseline table — ready to move to Objective 2 (defense strategy).